In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
# Define the directory path for the dataset
workshop_path = r'C:\Users\shomo\Desktop\workshop'

# Load training and test datasets from CSV files
train_df = pd.read_csv(os.path.join(workshop_path, 'train.csv'))
test_df = pd.read_csv(os.path.join(workshop_path, 'test.csv'))

# Print confirmation that data loading is complete
print("Loaded")

Loaded


In [ ]:
# Function to create new predictive features from raw data
def feature_engineering(df):
    # Create a copy to avoid modifying the original DataFrame
    data = df.copy()
    
    # Split PassengerId into group and individual number
    data[['Group', 'Number']] = data['PassengerId'].str.split('_', expand=True)
    data['Number'] = data['Number'].astype(int)
    
    # Calculate group size and check if passenger is traveling alone
    group_counts = data['Group'].map(data['Group'].value_counts())
    data['IsAlone'] = (group_counts == 1).astype(int)
    data['GroupSize'] = group_counts
    
    # Extract deck, side, and cabin number from Cabin field
    data['Deck'] = data['Cabin'].str[0]
    data['Side'] = data['Cabin'].str.split('/').str[2]
    data['CabinNum'] = data['Cabin'].str.split('/').str[1]
    data['CabinNum'] = pd.to_numeric(data['CabinNum'], errors='coerce')
    
    # Calculate total money spent across all amenities
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    data['TotalSpent'] = data[spend_cols].sum(axis=1)
    
    # Create VIP flag based on deck (A/B = VIP)
    data['IsVIP'] = (data['Deck'].isin(['A', 'B'])).astype(int)
    
    return data

In [ ]:
# Apply feature engineering to training dataset
train_processed = feature_engineering(train_df)

# Apply the same feature engineering to test dataset
test_processed = feature_engineering(test_df)

# Print completion message
print("Done")

Done


In [ ]:
# Ensure required variables exist before proceeding
if 'train_processed' not in locals():
    raise NameError("Please run cell 4 first")

# Define numerical and categorical feature lists
num_features = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','GroupSize','IsAlone','TotalSpent','IsVIP','CabinNum','Number']
cat_features = ['HomePlanet','CryoSleep','Destination','Deck','Side']
all_features = num_features + cat_features

# Prepare training and test datasets
X_train = train_processed[all_features].copy()
y_train = train_processed['Transported'].astype(int)
X_test = test_processed[all_features].copy()

# Store passenger IDs for submission
passenger_ids = test_processed['PassengerId']

# Print ready status
print("Ready")

Ready


In [ ]:
# Encode categorical features using LabelEncoder
for col in cat_features:
    # Combine train and test data for consistent encoding
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
    le = LabelEncoder()
    le.fit(combined)
    # Transform both datasets
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# Fill missing values with median strategy
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=all_features)
X_test = pd.DataFrame(imputer.transform(X_test), columns=all_features)

# Standardize features to mean=0, std=1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Print completion message
print("Preprocessing done")

Preprocessing done


In [ ]:
# Initialize Logistic Regression model with balanced class weights
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')

# Train the model on scaled training data
lr_model.fit(X_train_scaled, y_train)

# Perform 5-fold cross-validation to evaluate model performance
cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5)

# Print mean cross-validation accuracy
print(f"CV Accuracy: {cv_scores.mean():.4f}")

CV Accuracy: 0.7786


In [ ]:
# Generate test predictions and convert to boolean type
pred = lr_model.predict(X_test_scaled).astype(bool)

# Create submission DataFrame with PassengerId and predictions
sub = pd.DataFrame({'PassengerId': passenger_ids, 'Transported': pred})

# Save submission file to the target directory
sub.to_csv(os.path.join(workshop_path, 'submission_logistic_regression.csv'), index=False)

# Print save confirmation
print("Saved")

Saved
